# CDC-6 — Tombstones, deletes & replica identity

**Break → Detect → Fix → Prove.** A `DELETE` in Postgres is **two** CDC records: a **delete event**
(`op="d"`, `after=null`, key still in `before`) followed by a **tombstone** (a *null-value* record on
the same key) that lets Kafka **log compaction** physically drop the key. What the `before` image of a
delete/update actually *contains* is decided by the source table's **`REPLICA IDENTITY`**:

- **DEFAULT** → Postgres logs only the **primary key**, so `before` is **key-only**.
- **`REPLICA IDENTITY FULL`** → Postgres logs the **whole old row**, so `before` carries **all columns** (at the cost of a bigger WAL).

We register two connectors over two tables that differ *only* in replica identity, run the same
UPDATE/DELETE on each, and print the `before` images **side by side**.

**Prerequisite:** `make cdc-up`. **Isolation:** tables `cdc6_orders` / `cdc6_orders_full`, connectors
`cdc6-orders` / `cdc6-orders-full`. **Laptop-safe:** ~20-row tables, bounded reads, teardown at start **and** end.

> **Honesty:** CDC parts are deterministic (a delete *does* yield `d`+tombstone; identity *does* change
> `before`). Streaming has a small **decode lag**, so we `time.sleep(5)` after each DML and read with a
> bounded `max_ms` — never blocking the broker. **Suppressing** the tombstone
> (`tombstones.on.delete=false` / an `ExtractNewRecordState` SMT) is **described, not executed** — we keep
> the default on so the tombstone is visible.

In [ ]:
from common import cdc_helpers as cdc
import json, time

# Two tables/connectors that differ ONLY in REPLICA IDENTITY.
T_DEF,  C_DEF  = "cdc6_orders",      "cdc6-orders"        # DEFAULT replica identity (key-only before)
T_FULL, C_FULL = "cdc6_orders_full", "cdc6-orders-full"   # REPLICA IDENTITY FULL  (full old-row before)

print("Connect REST up:", cdc.connect_up(timeout=40))

# Clean slate so a re-run starts fresh (idempotent): delete connectors, drop slots/tables/topics.
cdc.teardown(C_DEF,  T_DEF)
cdc.teardown(C_FULL, T_FULL)
print("torn down any prior cdc6 connectors/tables/topics")

## Part A — DEFAULT replica identity: `before` is key-only

`seed_orders(..., replica_identity_full=False)` creates the table with Postgres's **default**
replica identity. We register the connector, let the initial snapshot land, then run a real
**UPDATE** and **DELETE** on one row and watch what the `before` image holds.

In [ ]:
n = cdc.seed_orders(T_DEF, n=20, replica_identity_full=False)
TOPIC_DEF = cdc.topic_name(T_DEF)
print(f"seeded public.{T_DEF} with {n} rows (DEFAULT replica identity); topic -> {TOPIC_DEF}")

reg = cdc.register_connector(C_DEF, cdc.debezium_pg_config(C_DEF, T_DEF, snapshot_mode="initial"))
print("register ->", reg["status"], "| state ->", cdc.wait_for_connector(C_DEF, timeout=60))

### Break it — UPDATE then DELETE one row

We change row `id=3` (status `NEW` → `SHIPPED`, bump the amount), then delete it. With default
identity, Postgres writes **only the key** of the old row to the WAL — so Debezium's `before` can
only ever be `{"id": 3}`.

In [ ]:
KEY = 3
cdc.pg_exec("UPDATE public.%s SET status='SHIPPED', amount=999.99 WHERE id=%%s" % T_DEF, (KEY,))
cdc.pg_exec("DELETE FROM public.%s WHERE id=%%s" % T_DEF, (KEY,))
time.sleep(5)   # bounded: let Debezium decode the WAL change events

events_def = cdc.read_cdc_events(TOPIC_DEF, max_ms=12000)
print("op counts (DEFAULT):", cdc.op_counts(events_def))   # expect r:20, u:1, d:1, tombstone:1

### Detect — inspect the `before` of the `u` and `d` events

`op_counts` should show a **`tombstone`** (the `None`-op null-value record) after the delete. Now
pull the `before` image of the update and the delete — both are **key-only** under default identity.

In [ ]:
u_def = next((e for e in events_def if e["op"] == "u"), None)
d_def = next((e for e in events_def if e["op"] == "d"), None)
n_tomb_def = cdc.op_counts(events_def).get("tombstone", 0)

print("UPDATE before (DEFAULT):", json.dumps(u_def["before"]) if u_def else None)
print("DELETE before (DEFAULT):", json.dumps(d_def["before"]) if d_def else None)
print("DELETE after  (DEFAULT):", json.dumps(d_def["after"])  if d_def else None, "  <- null on a delete")
print("tombstones after delete:", n_tomb_def, "  <- null-value record on the deleted key")

## Part B — `REPLICA IDENTITY FULL`: `before` is the whole old row

Same table shape, same connector settings — the **only** difference is
`seed_orders(..., replica_identity_full=True)`, which issues `ALTER TABLE … REPLICA IDENTITY FULL`.
Now Postgres writes the **entire old row** to the WAL on every UPDATE/DELETE, so Debezium fills
`before` with **all columns**.

In [ ]:
n = cdc.seed_orders(T_FULL, n=20, replica_identity_full=True)
TOPIC_FULL = cdc.topic_name(T_FULL)
print(f"seeded public.{T_FULL} with {n} rows (REPLICA IDENTITY FULL); topic -> {TOPIC_FULL}")

reg = cdc.register_connector(C_FULL, cdc.debezium_pg_config(C_FULL, T_FULL, snapshot_mode="initial"))
print("register ->", reg["status"], "| state ->", cdc.wait_for_connector(C_FULL, timeout=60))

In [ ]:
cdc.pg_exec("UPDATE public.%s SET status='SHIPPED', amount=999.99 WHERE id=%%s" % T_FULL, (KEY,))
cdc.pg_exec("DELETE FROM public.%s WHERE id=%%s" % T_FULL, (KEY,))
time.sleep(5)   # bounded decode lag

events_full = cdc.read_cdc_events(TOPIC_FULL, max_ms=12000)
print("op counts (FULL):", cdc.op_counts(events_full))   # expect r:20, u:1, d:1, tombstone:1

u_full = next((e for e in events_full if e["op"] == "u"), None)
d_full = next((e for e in events_full if e["op"] == "d"), None)
print("UPDATE before (FULL):", json.dumps(u_full["before"]) if u_full else None)
print("DELETE before (FULL):", json.dumps(d_full["before"]) if d_full else None)

## Prove it — the before-image table, key-only vs full

The core lesson in one table: the **same** UPDATE/DELETE on two tables that differ *only* in replica
identity. Default identity logs **only the key**; `FULL` logs the **complete old row**.

In [ ]:
def cols(before):
    return sorted(before.keys()) if isinstance(before, dict) else before

rows = [
    ("cdc6_orders (DEFAULT)", cols(u_def["before"]  if u_def  else None), cols(d_def["before"]  if d_def  else None)),
    ("cdc6_orders_full (FULL)", cols(u_full["before"] if u_full else None), cols(d_full["before"] if d_full else None)),
]
print(f'{"table (REPLICA IDENTITY)":28}  {"u.before columns":40}  d.before columns')
print("-" * 100)
for name, ub, db in rows:
    print(f"{name:28}  {str(ub):40}  {db}")

print()
print("DEFAULT  u.before:", json.dumps(u_def["before"])  if u_def  else None)
print("FULL     u.before:", json.dumps(u_full["before"]) if u_full else None)
print("tombstones — DEFAULT:", cdc.op_counts(events_def).get("tombstone", 0),
      "| FULL:", cdc.op_counts(events_full).get("tombstone", 0))

## Diagnose

- **Default identity = key-only `before`.** `REPLICA IDENTITY DEFAULT` makes Postgres write only the
  primary-key columns of the OLD row to the WAL — enough to *identify* the row (apply the delete /
  match the update), **not** enough to know its old values.
- **FULL = complete `before`.** `REPLICA IDENTITY FULL` writes the whole old row to the WAL, so
  `before` is complete — what audit trails, "previous value" rules, and non-PK joins need — at the
  cost of a **larger WAL on every UPDATE/DELETE**, paid on the source whether or not anyone reads it.
- **The tombstone.** A delete is a `d` event (`after=null`) **plus** a null-value record on the key.
  That null-value record is the **tombstone**; Kafka **compaction** uses it to physically GC the key
  (retained for `delete.retention.ms`, then dropped). It's emitted because
  `tombstones.on.delete=true` (the default in `debezium_pg_config`).

## Fix it / guidance

| Need | Setting |
|------|---------|
| Just apply deletes/updates (key is enough) | **default** `REPLICA IDENTITY` — minimal WAL |
| Old values (audit, previous status, diffs) | **`ALTER TABLE … REPLICA IDENTITY FULL`** — full old row, bigger WAL |
| Key on a **non-PK** unique column | `REPLICA IDENTITY USING INDEX <unique_idx>` (or `FULL`) |
| Let a compacted topic drop deleted keys | keep **`tombstones.on.delete=true`** (default) |
| Flatten the envelope / suppress the standalone tombstone | `ExtractNewRecordState` SMT + `delete.handling.mode` (+ often `tombstones.on.delete=false`) — *described, not run* |

**Downstream (forward-ref CDC-7).** A `d` event + tombstone means *"remove this key from the sink."*
In Spark → Iceberg that's a keyed, idempotent upsert:

```sql
MERGE INTO iceberg_catalog.mirror.orders t
USING cdc_batch s
ON t.id = s.id
WHEN MATCHED AND s.op = 'd' THEN DELETE
WHEN MATCHED               THEN UPDATE SET *
WHEN NOT MATCHED AND s.op <> 'd' THEN INSERT *
```

Tombstones (null values) are filtered before the MERGE — the `d` event already carries the delete.

**Suppressing the tombstone (described).** With `tombstones.on.delete=false`, Debezium emits the `d`
event but **no** trailing null-value record; an `ExtractNewRecordState` SMT
(`delete.handling.mode=rewrite`) instead flattens the envelope and adds a `__deleted=true` flag. Use
those when your sink consumes a flattened record rather than the raw envelope. We left the default on
above so the tombstone is observable.

## Takeaways & "in real production…"

- **A delete is two records** — handle the `d` (`after=null`) event *and* tolerate the **tombstone**
  (null value); don't let a null-value record crash a deserializer or get counted as a row. In a
  compacted topic the tombstone is what lets the key be GC'd.
- **`REPLICA IDENTITY` is a source-side decision with a downstream blast radius** — set it per table:
  **default** where the PK is enough to apply changes; **FULL** only where you truly need old values,
  and budget for the extra WAL it writes on every change.
- **Don't reach for FULL reflexively** — it's a permanent WAL tax. Consider `USING INDEX` when a
  non-PK unique key is what you join on.
- **Tie it together:** tombstones + a compacted topic (**KAF-4**) + a keyed idempotent `MERGE`
  (**CDC-7**) is how a lakehouse mirror stays an exact, delete-aware copy of its source.

## Teardown

In [ ]:
cdc.teardown(C_DEF,  T_DEF)     # delete connector, drop slot, drop table, delete topic
cdc.teardown(C_FULL, T_FULL)
print("slots now:", cdc.list_slots())